# VoiceGuard India - FINAL Generalized Model
KEY FIX: librosa.util.normalize(y) applied before all feature extraction.
This eliminates domain mismatch between HuggingFace FLAC and local WAV files.
Features: 54 (20 MFCC + 20 MFCC_std + 10 delta + 4 spectral) = fully domain-invariant.

In [ ]:
!conda install -y -c conda-forge ffmpeg


In [ ]:
!pip install pandas librosa xgboost scikit-learn fsspec pyarrow huggingface_hub soundfile


In [ ]:
import pandas as pd, librosa, numpy as np, io, pickle, xgboost as xgb, warnings
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from huggingface_hub import hf_hub_download
warnings.filterwarnings('ignore')

HF_TOKEN = 'hf_YOUR_TOKEN_HERE'

def extract_features(audio_input, from_bytes=True, sr=16000):
    try:
        if from_bytes:
            y, _ = librosa.load(io.BytesIO(audio_input['bytes']), sr=sr, duration=3.0)
        else:
            y, _ = librosa.load(audio_input, sr=sr, duration=3.0)
        if len(y) < 1000: return None
        y = librosa.util.normalize(y)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        mfcc_mean = np.mean(mfccs.T, axis=0)
        mfcc_std  = np.std(mfccs.T, axis=0)
        delta_mean = np.mean(librosa.feature.delta(mfccs).T, axis=0)[:10]
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr      = np.mean(librosa.feature.zero_crossing_rate(y))
        rolloff  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        rms      = np.mean(librosa.feature.rms(y=y))
        return np.hstack((mfcc_mean, mfcc_std, delta_mean, centroid, zcr, rolloff, rms))
    except: return None

X, y = [], []

for ds in [
    ('google/fleurs',          'parquet-data/hi_in/train-00000-of-00001.parquet', 'Google Fleurs', 1, 1500),
    ('SPRINGLab/IndicTTS-Hindi','data/train-00000-of-00010.parquet',               'IndicTTS',      1, 1200),
]:
    print('Downloading ' + ds[2] + ' (AUTHENTIC)...')
    try:
        lp = hf_hub_download(repo_id=ds[0], filename=ds[1], repo_type='dataset', token=HF_TOKEN)
        df = pd.read_parquet(lp)
        count = 0
        for _, row in tqdm(df.iterrows(), total=min(len(df), ds[4])):
            if count >= ds[4]: break
            if 'audio' in row and row['audio'] is not None:
                feat = extract_features(row['audio'])
                if feat is not None: X.append(feat); y.append(1); count += 1
        print('Got ' + str(count) + ' authentic samples')
    except Exception as e: print('ERROR: ' + str(e))

for sfile in [
    'data/train-00000-of-00004.parquet',
    'data/train-00001-of-00004.parquet',
    'data/train-00002-of-00004.parquet',
    'data/train-00003-of-00004.parquet',
]:
    print('Downloading Sarvam ' + sfile + ' (DEEPFAKE)...')
    try:
        lp = hf_hub_download(repo_id='abhirl/hindi-tts-dataset', filename=sfile, repo_type='dataset', token=HF_TOKEN)
        df = pd.read_parquet(lp)
        count = 0
        for _, row in tqdm(df.iterrows(), total=len(df)):
            if 'audio' in row and row['audio'] is not None:
                feat = extract_features(row['audio'])
                if feat is not None: X.append(feat); y.append(0); count += 1
        print('Got ' + str(count) + ' deepfake samples')
    except Exception as e: print('ERROR: ' + str(e))

X_final = np.array(X); y_final = np.array(y)
print('TOTAL: '+str(len(y_final))+' | Auth: '+str(int(np.sum(y_final==1)))+' | Fake: '+str(int(np.sum(y_final==0))))
print('Feature vector size: ' + str(X_final.shape[1]))
if len(np.unique(y_final)) < 2: print('FATAL: Missing one class!')
else: print('Ready to train!')


In [ ]:
if len(np.unique(y_final)) >= 2:
    X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.15, random_state=42, stratify=y_final)
    ratio = float(np.sum(y_train==1)) / max(float(np.sum(y_train==0)), 1.0)
    print('Class balance ratio: ' + str(round(ratio,2)))
    clf = xgb.XGBClassifier(
        n_estimators=600, max_depth=9, learning_rate=0.02,
        scale_pos_weight=ratio, subsample=0.85, colsample_bytree=0.85,
        min_child_weight=3, gamma=0.1,
        n_jobs=-1, random_state=42, eval_metric='logloss'
    )
    clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100, early_stopping_rounds=50)
    print('ACCURACY ON UNSEEN TEST DATA:')
    print(classification_report(y_test, clf.predict(X_test), target_names=['Deepfake','Authentic']))
    with open('voiceguard_final_model.pkl', 'wb') as fout: pickle.dump(clf, fout)
    print('SAVED: voiceguard_final_model.pkl')
    print('Download it -> rename to voiceguard_rf_model.pkl -> put in ml_engine/models/')
